# processing large ensembles

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import numpy as np
import os
import xarray as xr
import xcdat as xc
# import matplotlib.pyplot as plt
# from matplotlib.colors import BoundaryNorm as BM
import pandas as pd
# import matplotlib as mpl
# import matplotlib.ticker as mticker
import netCDF4
# import cartopy.crs as ccrs
# import cartopy.feature as cfeature
# from cartopy.util import add_cyclic_point
# from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

In [3]:
# from dask.distributed import Client
# client = Client(n_workers=4, threads_per_worker=2)
# client

In [4]:
from scipy import stats

In [5]:
# mpl.rcParams['font.family'] = 'Droid Sans'
# mpl.rcParams['font.size'] = 12
# # Edit axes parameters
# mpl.rcParams['axes.linewidth'] = 1.5
# # Tick properties
# mpl.rcParams['xtick.major.size'] = 5
# mpl.rcParams['xtick.minor.size'] = 3
# mpl.rcParams['xtick.major.width'] = 1
# mpl.rcParams['xtick.direction'] = 'out'
# mpl.rcParams['ytick.major.size'] = 5
# mpl.rcParams['ytick.minor.size'] = 3
# mpl.rcParams['ytick.major.width'] = 1
# mpl.rcParams['ytick.direction'] = 'out'

In [6]:
from dask.diagnostics import ProgressBar

### Functions needed for the analysis

In [7]:
from functions import preproc_funcs as funcs

In [8]:
from functions import xr_lowess

In [9]:
from statsmodels.tsa.seasonal import STL


def loess1d(x, period):
    x_copy = x.copy()
    res = STL(x_copy, period=period).fit()
    return res.trend


def loess3d(x, dim, period):
    return xr.apply_ufunc(loess1d, x, input_core_dims=[[dim]], output_core_dims=[[dim]], kwargs=dict(period=period), vectorize=True, dask="parallelized")

In [10]:
import glob
import multiprocessing as mp

In [11]:
# Function to find the first file in each model's r1* directory
def find_all_files(pattern):
    all_paths = glob.glob(pattern)
    model_files = {}
    for path in all_paths:
        # Adjust the split indices based on your folder structure
        path_parts = path.split('/')
        # Assuming the model name is at index 7 (adjust if needed)
        # model_identifier = path_parts[7] + '_' + path_parts[9][:-1]
        model_identifier = path_parts[8] + '_' + path_parts[10][:-1]
        if model_identifier not in model_files:
            model_files[model_identifier] = '/'.join(path_parts[:-1]) + '/*.nc'  # Store only the first file for each model
    return model_files


# Function to find the first file in each model's r1* directory
def find_all_files_extended(pattern):
    all_paths = glob.glob(pattern)
    model_files = {}
    for path in all_paths:
        # Adjust the split indices based on your folder structure
        path_parts = path.split('/')
        # Assuming the model name is at index 7 (adjust if needed)
        model_identifier = path_parts[8] + '_' + path_parts[10][:-1]
        if model_identifier not in model_files:
            model_files[model_identifier] = '/'.join(path_parts[:-1]) + '/*.nc'  # Store only the first file for each model
    return model_files



## find the files for a single model

In [49]:
model = 'MPI-ESM'

In [50]:
## cmip6
# ts_pattern_hist = f'/g/data/su28/MMLEAv2/atmosphere/monthly/tas/{model}/tas_Amon_{model}_historical*.nc'
# ts_pattern_ssp5 = f'/g/data/su28/MMLEAv2/atmosphere/monthly/tas/{model}/tas_Amon_{model}_ssp585*.nc'
# ts_pattern_ssp5 = f'/g/data/su28/MMLEAv2/atmosphere/monthly/tas/{model}/tas_Amon_{model}_historical_ssp585*.nc'

## cmip5
ts_pattern_rcp85 = f'/g/data/su28/MMLEAv2/atmosphere/monthly/tas/{model}/tas_Amon_{model}_historical_rcp85_*.nc'

In [51]:
# ## cmip6
# pr_pattern_hist = f'/g/data/su28/MMLEAv2/atmosphere/monthly/pr/{model}/pr_Amon_{model}_historical*.nc'
# pr_pattern_ssp5 = f'/g/data/su28/MMLEAv2/atmosphere/monthly/pr/{model}/pr_Amon_{model}_ssp585*.nc'
# pr_pattern_ssp5 = f'/g/data/su28/MMLEAv2/atmosphere/monthly/pr/{model}/pr_Amon_{model}_historical_ssp585*.nc'

## cmip5
pr_pattern_rcp85 = f'/g/data/su28/MMLEAv2/atmosphere/monthly/pr/{model}/pr_Amon_{model}_historical_rcp85_*.nc'

In [52]:
# ts_hist_files = sorted(glob.glob(ts_pattern_hist))
# ts_ssp5_files = sorted(glob.glob(ts_pattern_ssp5))

ts_rcp85_files = sorted(glob.glob(ts_pattern_rcp85))

In [53]:
# pr_hist_files = sorted(glob.glob(pr_pattern_hist))
# pr_ssp5_files = sorted(glob.glob(pr_pattern_ssp5))

pr_rcp85_files = sorted(glob.glob(pr_pattern_rcp85))

In [54]:
assert len(pr_hist_files) == len(pr_ssp5_files)

AssertionError: 

In [57]:
len(pr_rcp85_files), pr_rcp85_files

(100,
 ['/g/data/su28/MMLEAv2/atmosphere/monthly/pr/MPI-ESM/pr_Amon_MPI-ESM_historical_rcp85_r100i1p1_g025_185001-209912.nc',
  '/g/data/su28/MMLEAv2/atmosphere/monthly/pr/MPI-ESM/pr_Amon_MPI-ESM_historical_rcp85_r10i1p1_g025_185001-209912.nc',
  '/g/data/su28/MMLEAv2/atmosphere/monthly/pr/MPI-ESM/pr_Amon_MPI-ESM_historical_rcp85_r11i1p1_g025_185001-209912.nc',
  '/g/data/su28/MMLEAv2/atmosphere/monthly/pr/MPI-ESM/pr_Amon_MPI-ESM_historical_rcp85_r12i1p1_g025_185001-209912.nc',
  '/g/data/su28/MMLEAv2/atmosphere/monthly/pr/MPI-ESM/pr_Amon_MPI-ESM_historical_rcp85_r13i1p1_g025_185001-209912.nc',
  '/g/data/su28/MMLEAv2/atmosphere/monthly/pr/MPI-ESM/pr_Amon_MPI-ESM_historical_rcp85_r14i1p1_g025_185001-209912.nc',
  '/g/data/su28/MMLEAv2/atmosphere/monthly/pr/MPI-ESM/pr_Amon_MPI-ESM_historical_rcp85_r15i1p1_g025_185001-209912.nc',
  '/g/data/su28/MMLEAv2/atmosphere/monthly/pr/MPI-ESM/pr_Amon_MPI-ESM_historical_rcp85_r16i1p1_g025_185001-209912.nc',
  '/g/data/su28/MMLEAv2/atmosphere/monthl

In [58]:
# len(pr_ssp5_files), pr_ssp5_files

In [59]:
def open_member_cmip6(hist_file, ssp_file, ens_idx):
    ds_hist = xr.open_mfdataset(hist_file, use_cftime=True)
    ds_ssp = xr.open_mfdataset(ssp_file, use_cftime=True)

    # concat in time
    ds = xr.concat([ds_hist, ds_ssp], dim='time')

    # enforce monotonic time and drop overlaps if any
    ds = ds.sortby('time')
    _, index = np.unique(ds.time, return_index=True)
    ds = ds.isel(time = index)

    # add_ensemble dimension
    ds = ds.assign_coords(model = ens_idx).expand_dims('model')

    return ds


def open_member_cmip6_alternate(ssp_file, ens_idx):
    ds = xr.open_mfdataset(ssp_file, use_cftime=True)

    # concat in time

    # enforce monotonic time and drop overlaps if any
    ds = ds.sortby('time')
    _, index = np.unique(ds.time, return_index=True)
    ds = ds.isel(time = index)

    # add_ensemble dimension
    ds = ds.assign_coords(model = ens_idx).expand_dims('model')

    return ds


def open_member_cmip5(rcp_file, ens_idx):
    ds = xr.open_mfdataset(rcp_file, use_cftime=True)

    # enforce monotonic time and drop overlaps if any
    ds = ds.sortby('time')
    _, index = np.unique(ds.time, return_index=True)
    ds = ds.isel(time = index)

    # add_ensemble dimension
    ds = ds.assign_coords(model = ens_idx).expand_dims('model')

    return ds

In [60]:
# members = []
# for i, (hf, sf) in enumerate(zip(ts_hist_files, ts_ssp5_files)):
#     members.append(open_member_cmip6(hf, sf, i))


members = []
for i, rf in enumerate(ts_rcp85_files):
    members.append(open_member_cmip5(rf, i))

ERROR 1: PROJ: proj_create_from_database: Open of /g/data/xp65/public/./apps/med_conda/envs/analysis3-25.09/share/proj failed


In [61]:
ds = xr.concat(members, dim='model')
ds

<xarray.Dataset> Size: 25GB
Dimensions:    (model: 100, time: 3000, bnds: 2, lat: 72, lon: 144)
Coordinates:
  * time       (time) object 24kB 1850-01-16 12:00:00 ... 2099-12-16 12:00:00
  * lon        (lon) float64 1kB 1.25 3.75 6.25 8.75 ... 351.2 353.8 356.2 358.8
  * lat        (lat) float64 576B -88.75 -86.25 -83.75 ... 83.75 86.25 88.75
    height     float64 8B 2.0
  * model      (model) int64 800B 0 1 2 3 4 5 6 7 8 ... 92 93 94 95 96 97 98 99
Dimensions without coordinates: bnds
Data variables:
    time_bnds  (model, time, bnds) object 5MB dask.array<chunksize=(1, 3000, 2), meta=np.ndarray>
    tas        (model, time, lat, lon) float64 25GB dask.array<chunksize=(1, 3000, 72, 144), meta=np.ndarray>
Attributes: (12/29)
    CDI:                    Climate Data Interface version 2.0.2 (https://mpi...
    Conventions:            CF-1.4
    source:                 MPI-ESM 1.1.00p2
    institution:            Max-Planck-Institut fuer Meteorologie
    history:                Mon Nov 21 18:05:25 2022: cdo -O -b F64 -remapcon...
    institute_id:           MPI-M
    ...                     ...
    title:                  MPI-ESM model output prepared for CMIP5 historical
    parent_experiment:      pre-industrial control
    modeling_realm:         atmos
    realization:            100
    cmor_version:           2.9.1
    CDO:                    Climate Data Operators version 2.0.1 (https://mpi...

In [62]:
tas = ds["tas"]
# tas = tas.sel(time = slice('1850-01-01', '2100-12-31'))
tas = tas.chunk(dict(model=8, time=120, lat=50, lon=50))

In [63]:
tas_ann = tas.resample(time = 'AS-JUN').mean('time')

In [64]:
with ProgressBar():
    tas_ann_loaded = tas_ann.load()

[###                                     ] | 9% Completed | 26.38 sms

[########################################] | 100% Completed | 147.19 s


In [65]:
# members = []
# for i, (hf, sf) in enumerate(zip(pr_hist_files, pr_ssp5_files)):
#     members.append(open_member_cmip6(hf, sf, i))


members = []
for i, rf in enumerate(pr_rcp85_files):
    members.append(open_member_cmip5(rf, i))

In [66]:
ds = xr.concat(members, dim='model')
ds

<xarray.Dataset> Size: 25GB
Dimensions:    (model: 100, time: 3000, bnds: 2, lat: 72, lon: 144)
Coordinates:
  * time       (time) object 24kB 1850-01-16 12:00:00 ... 2099-12-16 12:00:00
  * lon        (lon) float64 1kB 1.25 3.75 6.25 8.75 ... 351.2 353.8 356.2 358.8
  * lat        (lat) float64 576B -88.75 -86.25 -83.75 ... 83.75 86.25 88.75
  * model      (model) int64 800B 0 1 2 3 4 5 6 7 8 ... 92 93 94 95 96 97 98 99
Dimensions without coordinates: bnds
Data variables:
    time_bnds  (model, time, bnds) object 5MB dask.array<chunksize=(1, 3000, 2), meta=np.ndarray>
    pr         (model, time, lat, lon) float64 25GB dask.array<chunksize=(1, 3000, 72, 144), meta=np.ndarray>
Attributes:
    CDI:          Climate Data Interface version 2.0.2 (https://mpimet.mpg.de...
    Conventions:  CF-1.6
    history:      Tue Apr 12 10:09:16 2022: cdo -O -b F64 -remapcon2,/glade/c...
    CDO:          Climate Data Operators version 2.0.1 (https://mpimet.mpg.de...

In [67]:
pr = ds["pr"]
# pr = pr.sel(time = slice('1850-01-01', '2100-12-31'))
pr = pr.chunk(dict(model=8, time=120, lat=50, lon=50))

In [68]:
pr_ann = pr.resample(time = 'AS-JUN').mean('time')

In [69]:
with ProgressBar():
    pr_ann_loaded = pr_ann.load()

[######                                  ] | 15% Completed | 22.37 ss

[########################################] | 100% Completed | 145.02 s


In [70]:
pr_ann_loaded

<xarray.DataArray 'pr' (model: 100, time: 251, lat: 72, lon: 144)> Size: 2GB
array([[[[2.07201605e-06, 2.09071051e-06, 2.12971979e-06, ...,
          1.98508853e-06, 2.02318964e-06, 2.07124542e-06],
         [2.37834734e-06, 2.37325788e-06, 2.45971396e-06, ...,
          2.26710272e-06, 2.40953816e-06, 2.46037766e-06],
         [3.47808927e-06, 3.25907765e-06, 2.92906725e-06, ...,
          3.42190403e-06, 3.56829792e-06, 3.59418043e-06],
         ...,
         [1.06643736e-05, 9.83079849e-06, 9.59365448e-06, ...,
          1.16551991e-05, 1.15743843e-05, 1.11666766e-05],
         [9.10031431e-06, 9.18271085e-06, 9.20557645e-06, ...,
          9.36860702e-06, 9.16019771e-06, 9.01356038e-06],
         [4.75260495e-06, 4.66417212e-06, 4.57946064e-06, ...,
          5.19567493e-06, 5.02558232e-06, 4.87455291e-06]],

        [[1.72277105e-06, 1.72337896e-06, 1.73000653e-06, ...,
          1.69558434e-06, 1.70661053e-06, 1.71120259e-06],
         [2.12498817e-06, 2.10722403e-06, 2.05637579e-06, ...,
          2.17940266e-06, 2.17932606e-06, 2.16803446e-06],
         [2.23076835e-06, 2.20064923e-06, 2.04716446e-06, ...,
          2.69920080e-06, 2.55089378e-06, 2.43173414e-06],
...
         [9.79325163e-06, 9.94526893e-06, 9.99847475e-06, ...,
          1.01657379e-05, 1.00168517e-05, 9.79759689e-06],
         [1.09399588e-05, 1.11819288e-05, 1.13461818e-05, ...,
          1.08181186e-05, 1.08061023e-05, 1.08317848e-05],
         [8.77319053e-06, 8.73584666e-06, 8.74245306e-06, ...,
          8.93108866e-06, 8.78065017e-06, 8.76122677e-06]],

        [[2.18663668e-06, 2.20095628e-06, 2.21832767e-06, ...,
          2.15943238e-06, 2.17964526e-06, 2.18481556e-06],
         [2.42516087e-06, 2.46088134e-06, 2.38980592e-06, ...,
          2.48941702e-06, 2.52932148e-06, 2.47061894e-06],
         [2.33171287e-06, 2.18633959e-06, 2.04819971e-06, ...,
          2.60548323e-06, 2.53900813e-06, 2.46786473e-06],
         ...,
         [1.94883700e-05, 1.92842454e-05, 1.91662419e-05, ...,
          2.07911263e-05, 2.01848112e-05, 1.97590907e-05],
         [1.89047341e-05, 1.89959513e-05, 1.89744239e-05, ...,
          1.87288940e-05, 1.88212733e-05, 1.88338553e-05],
         [1.43248899e-05, 1.44196929e-05, 1.46020268e-05, ...,
          1.40926004e-05, 1.41162123e-05, 1.42276884e-05]]]])
Coordinates:
  * lon      (lon) float64 1kB 1.25 3.75 6.25 8.75 ... 351.2 353.8 356.2 358.8
  * lat      (lat) float64 576B -88.75 -86.25 -83.75 ... 83.75 86.25 88.75
  * model    (model) int64 800B 0 1 2 3 4 5 6 7 8 ... 91 92 93 94 95 96 97 98 99
  * time     (time) object 2kB 1849-06-01 00:00:00 ... 2099-06-01 00:00:00
Attributes:
    standard_name:   precipitation_flux
    long_name:       precipitation_flux
    units:           kg m-2 s-1
    code:            4
    table:           128
    comment:         Created using NCL code
    cell_methods:    time: mean (interval: 1 month)
    history:         No change
    original_units:  kg m-2 s-1
    original_name:   total precipitation

In [71]:
tas_ann_loaded.to_netcdf(f'/g/data/if69/as8561/data/enso_tele_robustness/smiles/{model}_ts.nc')
pr_ann_loaded.to_netcdf(f'/g/data/if69/as8561/data/enso_tele_robustness/smiles/{model}_pr.nc')

In [139]:
# tas_anom = funcs.calc_anom(tas_ann_loaded, tas_ann_loaded.sel(time = slice('1960', '1990')))
# tas_det = funcs.detrend_rolling_window(tas_anom, window_size=15)

In [140]:
# pr_det = funcs.calc_anom(pr_ann_loaded, pr_ann_loaded.sel(time = slice('1960', '1990')))
# pr_det = funcs.detrend_rolling_window(pr_det, window_size=15)
# pr_std = pr_det/pr_det.std('time')